# Chapter 2: NULL Handling, Aggregation, Subqueries and Comparison Operators
**Reference:** Cheat Sheet Part 1 (Section 11) + Cheat Sheet Part 2 (Sections 19, 20, 21) + QnA Part 2 (Reference Chapter 4)

---

## 1. Understanding NULL

### What is NULL?

- NULL means **unknown** or **missing data** — it is not zero, not an empty string, not a blank space
- It means: *"we don't know what value goes here"*
- NULL is not a value itself — it represents the **absence** of a value

### NULL in Comparisons

Any comparison you write with NULL does **not** give TRUE or FALSE. It gives a third result called **UNKNOWN**.

- `100 = NULL` → UNKNOWN
- `100 <> NULL` → UNKNOWN
- `NULL = NULL` → UNKNOWN
- `col IS NULL` → TRUE (if col is NULL) — this is the **only correct way** to check for NULL
- `col IS NOT NULL` → TRUE (if col has a value)

> **Why this matters:** SQL only keeps rows where the WHERE condition is TRUE. UNKNOWN is treated the same as FALSE — so rows with NULL in a filtered column get **silently excluded** without any error or warning.

### NULL in Arithmetic

Any arithmetic operation that touches NULL will always produce NULL — no exceptions.

- `100 + NULL` → NULL
- `NULL * 5` → NULL
- `NULL + 0` → NULL (even adding zero to NULL gives NULL)
- `100 / NULL` → NULL

> **Interview Point:** "What does `SUM(col)` return if every value in that column is NULL?"
> The answer is **NULL**, not zero. `COUNT(*)` would still return the row count, but `SUM` and `AVG` of all-NULL values return NULL. This trips up a lot of people who assume SQL converts NULL to zero automatically.

### NULL in Aggregate Functions

This is one of the most important rules to know:

- `SUM`, `AVG`, `MIN`, `MAX`, `COUNT(col)` — all **ignore** NULL values automatically
- `COUNT(*)` — counts every row, **including** rows where columns are NULL
- `AVG(col)` divides by the count of **non-NULL** values, not by total rows

```sql
-- If some profit values are NULL:
-- AVG(profit) = SUM of non-null profits / count of non-null profits only
-- Rows with NULL profit are not included in the calculation at all
SELECT
    COUNT(*)       AS total_rows,
    COUNT(profit)  AS rows_with_profit_value,
    AVG(profit)    AS avg_profit
FROM orders;
```

---

## 2. NULL Handling Functions in MSSQL

### ISNULL

- `ISNULL(expression, replacement)` — if the expression is NULL, return the replacement value instead
- Takes exactly **2 arguments**

```sql
-- If city is NULL, show 'Unknown' instead
SELECT
    customer_name,
    ISNULL(city, 'Unknown') AS city
FROM orders;
```

### COALESCE

- `COALESCE(expr1, expr2, expr3, ...)` — returns the **first non-NULL** value from left to right
- Takes **2 or more** arguments
- This is ANSI standard SQL — works the same in every database

```sql
-- Returns city if not NULL, otherwise state, otherwise 'Unknown'
SELECT
    customer_name,
    COALESCE(city, state, 'Unknown') AS resolved_location
FROM orders;
```

> **Best Practice:** Prefer `COALESCE` over `ISNULL` when you want clean, portable code. `ISNULL` is MSSQL-specific. `COALESCE` is ANSI standard and works across every major database engine.

### NULLIF

- `NULLIF(expr1, expr2)` — returns NULL if both expressions are equal, otherwise returns expr1
- Useful when you want to **convert a specific value into NULL** (often used to prevent divide-by-zero errors)

```sql
-- Prevent divide-by-zero: if quantity = 0, NULLIF returns NULL
-- Dividing by NULL returns NULL instead of throwing an error
SELECT
    order_id,
    sales / NULLIF(quantity, 0) AS price_per_unit
FROM orders;
```

### NULL Arithmetic Fix Using ISNULL or COALESCE

When you need to add columns that might be NULL and want 0 instead of NULL:

```sql
-- Without fix: if either profit or sales is NULL, result is NULL
SELECT profit + sales FROM orders;

-- With fix: treat NULL as 0 before adding
SELECT ISNULL(profit, 0) + ISNULL(sales, 0) AS combined
FROM orders;
```

---

## 3. COUNT Variants — The Most Misunderstood Part

### The Three Forms

- `COUNT(*)` — counts **every row**, including rows where all columns are NULL
- `COUNT(col)` — counts only rows where that specific column is **NOT NULL**
- `COUNT(DISTINCT col)` — counts **unique non-NULL** values in that column

```sql
SELECT
    COUNT(*)                AS count_all_rows,
    COUNT(order_id)         AS count_order_id,
    COUNT(postal_code)      AS count_postal_code,
    COUNT(DISTINCT city)    AS count_unique_cities
FROM orders;
```

If `postal_code` has some NULL values, `COUNT(postal_code)` will be **less than** `COUNT(*)`. That difference tells you exactly how many rows have missing postal codes.

> **Interview Point — COUNT(1) myth:**
> Very common question: "Is `COUNT(1)` faster than `COUNT(*)`?"
> Answer: **No.** In SQL Server, they compile to the exact same execution plan. Neither evaluates any actual column — both just count rows. The real distinction that matters is `COUNT(*)` versus `COUNT(column_name)` — the column version genuinely does more work, checking each row for NULL. Confidently correcting this myth shows real understanding of how the engine works.

---

## 4. Aggregation Deep Dive

### All Aggregate Functions and Their NULL Behaviour

- `COUNT(*)` — counts all rows, NULLs included
- `COUNT(col)` — skips NULLs
- `SUM(col)` — adds up non-NULL values, skips NULLs
- `AVG(col)` — averages non-NULL values only (denominator is non-NULL count, not row count)
- `MIN(col)` — finds smallest non-NULL value
- `MAX(col)` — finds largest non-NULL value

### Computing AVG Two Ways

```sql
-- Standard way
SELECT AVG(sales) AS avg_sales FROM orders;

-- Manual way — same result, but you must use COUNT(sales), not COUNT(*)
-- This is because AVG ignores NULLs, so the denominator must also ignore NULLs
SELECT SUM(sales) / COUNT(sales) AS avg_sales FROM orders;
```

### Aggregation Examples from the QnA Bank

**Total profit, first order date, and latest order date per category:**

```sql
-- MIN and MAX on date columns give you the earliest and latest dates automatically
SELECT
    category,
    SUM(profit)       AS total_profit,
    MIN(order_date)   AS first_order,
    MAX(order_date)   AS latest_order
FROM orders
GROUP BY category;
```

**Total sales by region and ship_mode for year 2020:**

```sql
SELECT
    region,
    ship_mode,
    SUM(sales) AS total_sales
FROM orders
WHERE YEAR(order_date) = 2020
GROUP BY region, ship_mode;
```

**Total distinct products in each category:**

```sql
SELECT
    category,
    COUNT(DISTINCT product_id) AS total_products
FROM orders
GROUP BY category;
```

**Top 5 sub-categories in West region by total quantity sold:**

```sql
SELECT TOP 5
    sub_category,
    SUM(quantity) AS quantity_sold
FROM orders
WHERE region = 'West'
GROUP BY sub_category
ORDER BY SUM(quantity) DESC;
```

**More one-liners:**

```sql
-- Average sales per ship mode
SELECT ship_mode, AVG(sales) AS avg_sales FROM orders GROUP BY ship_mode;

-- Maximum profit per region
SELECT region, MAX(profit) AS max_profit FROM orders GROUP BY region;

-- Minimum discount per category
SELECT category, MIN(discount) AS min_discount FROM orders GROUP BY category;

-- Number of orders per year
SELECT YEAR(order_date) AS order_year, COUNT(*) AS no_of_orders
FROM orders
GROUP BY YEAR(order_date)
ORDER BY order_year;
```

---

## 5. GROUP BY Scoping Trap

### The Problem

When you use `GROUP BY`, every aggregate function in `SELECT` or `HAVING` is calculated **per group**, not across the whole table. This becomes a trap when the question asks for a comparison against a **global** value.

**Question:** Find sub-categories where average profit is greater than half of the **global** maximum profit.

**Wrong Approach:**

```sql
-- WRONG
SELECT sub_category
FROM orders
GROUP BY sub_category
HAVING AVG(profit) > MAX(profit) / 2;
```

Why wrong? `MAX(profit)` here is calculated **within each sub_category group** because of `GROUP BY`. It is not the overall maximum across the entire table. You are comparing each group's average against that same group's maximum — which is not what the question asks.

**Correct Approach:**

```sql
-- CORRECT
SELECT sub_category
FROM orders
GROUP BY sub_category
HAVING AVG(profit) > (SELECT MAX(profit) FROM orders) / 2;
```

The subquery `(SELECT MAX(profit) FROM orders)` has no `GROUP BY` of its own, so it runs across the **entire table** and returns a single number — the true global maximum. Now each group's average is compared against that one global number.

> **Interview Point:** Any time the requirement says "global average," "overall maximum," "company-wide total" alongside a `GROUP BY`, you need a **subquery in HAVING** to keep that global calculation separate from the grouping scope. This is one of the most common wrong answers in SQL interviews.

---

## 6. Subquery Types

A subquery is a `SELECT` statement written **inside another query**, enclosed in parentheses.

### Type 1: Scalar Subquery

- Returns **exactly one value** — one row, one column
- Can be used anywhere you'd normally put a single value
- Used with `=`, `>`, `<`, `>=`, `<=`

```sql
-- Employees earning more than the company-wide average salary
SELECT emp_name, salary
FROM employee
WHERE salary > (SELECT AVG(salary) FROM employee);
```

```sql
-- Scalar subquery inside SELECT list — adds a calculated column from another query
SELECT
    customer_name,
    (SELECT SUM(sales) FROM orders o2 WHERE o2.customer_id = o1.customer_id) AS total_customer_sales
FROM orders o1
GROUP BY customer_id, customer_name;
```

### Type 2: Multi-row Subquery

- Returns **many rows**, single column
- Runs **once** for the entire outer query — it is not re-evaluated per row
- Used with `IN`, `NOT IN`, `EXISTS`, `NOT EXISTS`

```sql
-- Orders placed by customers who have also ordered from Mumbai
SELECT order_id, customer_name
FROM orders
WHERE customer_id IN (
    SELECT DISTINCT customer_id
    FROM orders
    WHERE city = 'Mumbai'
);
```

### Type 3: Correlated Subquery

- The inner query **references a column from the outer query**
- Because of this reference, the inner query **re-runs once for every single row** the outer query processes
- Slower than non-correlated subqueries but very powerful for row-by-row comparisons

```sql
-- Employees earning more than their OWN department's average
-- The subquery re-runs for each employee row because t1.dept_id changes each time
SELECT emp_name, salary, dept_id
FROM employee t1
WHERE salary > (
    SELECT AVG(salary)
    FROM employee t2
    WHERE t2.dept_id = t1.dept_id   -- t1.dept_id is the reference to the outer query
);
```

### Where Can Subqueries Be Placed?

- In `WHERE` — most common
- In `HAVING` — for filtering groups (shown in the scoping trap section above)
- In `SELECT` — to add a calculated column per row (scalar only)
- In `FROM` — treated as a derived table (covered in later chapters)

---

## 7. EXISTS vs IN

### EXISTS

- Checks whether a subquery **returns at least one row**
- Does not care about the actual values returned — only cares about existence
- The `SELECT 1` inside `EXISTS` is a convention — the value selected is completely ignored

```sql
-- Find all orders that have been returned
SELECT o.order_id, o.customer_name, o.sales
FROM orders o
WHERE EXISTS (
    SELECT 1
    FROM returns r
    WHERE r.order_id = o.order_id
);
```

### IN

- Checks whether a value **matches any value in a list or subquery result**

```sql
-- Same result, using IN instead
SELECT order_id, customer_name, sales
FROM orders
WHERE order_id IN (
    SELECT order_id FROM returns
);
```

### Key Differences Between EXISTS and IN

**Performance:**
- `EXISTS` stops searching as soon as it finds the first matching row — called "short circuit"
- `IN` loads all values from the subquery first, then checks each row against that complete list
- For large datasets, `EXISTS` is generally faster

**NULL handling:**
- `EXISTS` and `NOT EXISTS` are completely safe with NULLs
- `NOT IN` breaks silently when the subquery returns any NULL (covered in next section)

**Best use:**
- Use `EXISTS` when asking "does a related row exist?" — especially with correlated subqueries
- Use `IN` when checking against a small, known list or simple non-correlated subquery

> **Interview Point:** `SELECT 1` inside `EXISTS` — interviewers sometimes ask why you wrote `SELECT 1` and not `SELECT *`. The answer is: the value selected inside `EXISTS` is completely irrelevant — SQL Server ignores it. Writing `SELECT 1` is just a convention that communicates "I only care about whether a row exists, not what's in it." There is no performance difference.

---

## 8. NOT EXISTS vs NOT IN — The NULL Trap

### NOT EXISTS (Safe)

```sql
-- Orders that have NEVER been returned — using NOT EXISTS (recommended)
SELECT o.order_id, o.customer_name, o.sales
FROM orders o
WHERE NOT EXISTS (
    SELECT 1
    FROM returns r
    WHERE r.order_id = o.order_id
);
```

`NOT EXISTS` is always safe. If `returns.order_id` is NULL for some row, `NOT EXISTS` simply treats it as "no match found for this outer row," which is exactly the correct behaviour.

### NOT IN (Dangerous with NULLs)

```sql
-- This looks correct but silently returns ZERO ROWS if returns.order_id has any NULLs
SELECT order_id, customer_name
FROM orders
WHERE order_id NOT IN (
    SELECT order_id FROM returns
);
```

### Why NOT IN Breaks

`order_id NOT IN ('CA-001', 'CA-002', NULL)` expands to:

```
order_id <> 'CA-001'
AND order_id <> 'CA-002'
AND order_id <> NULL
```

The last condition — `order_id <> NULL` — always evaluates to **UNKNOWN**, not TRUE or FALSE. So the entire condition becomes UNKNOWN, and the row is excluded. This happens **for every single row in the outer table** — resulting in zero rows returned, with no error and no warning.

### The Fix for NOT IN

If you must use `NOT IN`, filter NULLs out of the subquery explicitly:

```sql
SELECT order_id, customer_name
FROM orders
WHERE order_id NOT IN (
    SELECT order_id FROM returns WHERE order_id IS NOT NULL
);
```

> **Interview Point — highest frequency trap question at senior level:**
> Expected answer has three parts:
> 1. **Name the problem:** `NOT IN` with a NULL in the subquery result silently returns zero rows
> 2. **Explain why:** because `x <> NULL` evaluates to UNKNOWN, and UNKNOWN is treated as FALSE in WHERE
> 3. **Give the fix:** prefer `NOT EXISTS` — it has no NULL sensitivity at all

---

## 9. ANY vs ALL

MSSQL fully supports `ANY` and `ALL` natively.

### ANY (same as SOME)

- `> ANY` means: greater than **at least one** value in the set
- Equivalent to: just beat the smallest (minimum) value in the set

```sql
-- Employees earning more than at least one person in the Sales department
SELECT emp_name, salary
FROM employee
WHERE salary > ANY (SELECT salary FROM employee WHERE dept_id = 1);
```

### ALL

- `> ALL` means: greater than **every** value in the set
- Equivalent to: must beat the largest (maximum) value in the set

```sql
-- Employees earning more than everyone in the Sales department
SELECT emp_name, salary
FROM employee
WHERE salary > ALL (SELECT salary FROM employee WHERE dept_id = 1);
```

### Memory Trick

- **"greater than ANY"** = just beat the weakest one = beat the `MIN`
- **"greater than ALL"** = beat every single one = beat the `MAX`

### ALL the Combinations

- `> ANY` — greater than at least one → beats the minimum
- `> ALL` — greater than all → beats the maximum
- `< ANY` — less than at least one → smaller than the maximum
- `< ALL` — less than all → smaller than the minimum
- `= ANY` — equals at least one → same as `IN`
- `<> ALL` — not equal to any → same as `NOT IN` (apply NULL caution)

> **Interview Point:** `= ANY` and `IN` produce identical results and identical execution plans in SQL Server. `<> ALL` and `NOT IN` also produce identical results — and both share the same NULL trap. Knowing this equivalence shows depth beyond just syntax memorization.

---

## 10. JOIN vs Subquery — When to Use Which

### Simple Rule

- Use a **JOIN** when you need **columns from more than one table** in your output
- Use a **Subquery** when you need data from another table **only for filtering or calculation** — you don't need to display that table's columns

```sql
-- JOIN: you want to SEE columns from both orders and returns in the result
SELECT o.order_id, o.customer_name, o.sales, r.return_reason
FROM orders o
INNER JOIN returns r ON o.order_id = r.order_id;
```

```sql
-- Subquery: you only want orders data, and use returns just to filter
SELECT order_id, customer_name, sales
FROM orders
WHERE order_id IN (SELECT order_id FROM returns);
```

### When a Correlated Subquery Can't Be Easily Replaced by a JOIN

Correlated subqueries that do per-row calculations (like "compare this row's salary against its own department's average") are harder to rewrite as a JOIN and often cleaner to leave as a subquery:

```sql
-- Clean as a correlated subquery — hard to express the same way with just a JOIN
SELECT emp_name, salary
FROM employee t1
WHERE salary > (SELECT AVG(salary) FROM employee t2 WHERE t2.dept_id = t1.dept_id);
```

> **Interview Point:** "Can a subquery always be rewritten as a JOIN?"
> Answer: Non-correlated subqueries can almost always be rewritten as a JOIN and often perform better that way. Correlated subqueries are harder — sometimes impossible — to cleanly express as a JOIN, and may actually be clearer as a subquery. The real decision is **readability plus performance** — check the execution plan rather than defaulting to one approach.

---

## 11. Quick Summary for Revision

### NULL Rules

- NULL means unknown — not zero, not blank
- Any comparison with NULL gives UNKNOWN — never TRUE or FALSE
- Any arithmetic with NULL gives NULL
- Only `IS NULL` and `IS NOT NULL` work correctly to check for NULL
- Aggregate functions (`SUM`, `AVG`, `MIN`, `MAX`, `COUNT(col)`) all ignore NULLs
- `COUNT(*)` does not ignore NULLs — it counts every row

### Function Quick Recall

- `ISNULL(col, 'default')` — 2 arguments, MSSQL specific
- `COALESCE(col1, col2, 'default')` — 2+ arguments, returns first non-NULL, works everywhere
- `NULLIF(col, 0)` — returns NULL if the two values are equal, used to prevent divide-by-zero

### Subquery Types Quick Recall

- **Scalar** — returns one value, used with `=`, `>`, `<`
- **Multi-row** — returns many values, used with `IN`, `EXISTS`
- **Correlated** — re-runs per outer row, references outer query

### EXISTS vs IN Quick Recall

- `EXISTS` — short circuits on first match, safe with NULLs
- `IN` — loads all values first, faster on small lists
- `NOT EXISTS` — always safe, preferred for anti-joins
- `NOT IN` — breaks silently if subquery returns any NULL

### ANY vs ALL Quick Recall

- `> ANY` = `> MIN` of the set
- `> ALL` = `> MAX` of the set
- `= ANY` = same as `IN`
- `<> ALL` = same as `NOT IN`

---